# 04 — Model Evaluation & Insights

**Fish Habitat / PFZ — Problem Statement B**

Training produced numbers. This notebook asks whether they mean anything, and
what the model has actually learned.

| Section | Question |
|---|---|
| 1. Skill | How well does it discriminate, on unseen water? |
| 2. Curves | Where does it trade precision for recall? |
| 3. Boyce | Does it rank habitat sensibly *without* true absences? |
| 4. SHAP | Which features drive predictions? |
| 5. Response curves | Do those relationships match known ecology? |
| 6. MESS | Where is it extrapolating rather than interpolating? |
| 7. Suitability map | What does it actually predict, spatially? |
| 8. Limitations | What should *not* be claimed from this? |

Section 8 is not boilerplate. The most useful output of an evaluation is a clear
statement of where the model should not be trusted.

In [ ]:
import sys, warnings
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
sys.path.insert(0, str(PROJECT_ROOT))
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from marine_ml import config, fusion, viz
from marine_ml.sources import copernicus, gebco
from marine_ml.validation import splits, metrics
from fish_habitat_prediction.src import features as feature_lib

viz.use_house_style()

REGION = config.NORTH_INDIAN_OCEAN
START, END = config.HABITAT_START, config.HABITAT_END

artifact = joblib.load(config.MODELS_DIR / "fish_habitat.joblib")
models = artifact["models"]
niche = artifact["thermal_niche"]
ensemble_weights = artifact["ensemble_weights"]
model_columns = artifact["feature_columns"]

featured = fusion.read_feature_store("fish_habitat_points")
frame = feature_lib.drop_unusable_rows(featured, model_columns)

fold_scores = pd.read_csv(config.REPORTS_DIR / "fish_habitat_fold_scores.csv")
holdout = pd.read_csv(config.REPORTS_DIR / "fish_habitat_holdout.csv")

print(f"models        : {sorted(models)}")
print(f"weights       : { {k: round(v, 3) for k, v in ensemble_weights.items()} }")
print(f"evaluation set: {len(frame)} rows, {int(frame.presence.sum())} presences")

## 1. Skill summary

Two views of the same models. The cross-validated numbers average over five
held-out blocks; the held-out numbers are one specific block never used for
model selection.

In [ ]:
comparison = pd.DataFrame({
    "CV ROC-AUC": fold_scores.groupby("model").roc_auc.mean(),
    "CV PR-AUC": fold_scores.groupby("model").pr_auc.mean(),
    "CV TSS": fold_scores.groupby("model").tss.mean(),
    "CV Boyce": fold_scores.groupby("model").boyce.mean(),
    "holdout ROC-AUC": holdout.set_index("model").roc_auc,
    "holdout TSS": holdout.set_index("model").tss,
    "holdout Boyce": holdout.set_index("model").boyce,
}).round(3)
comparison

### What these metrics mean here

- **ROC-AUC** — probability a random presence scores above a random background
  point. Optimistic under class imbalance, so it is reported but not led with.
- **PR-AUC** — the honest one when positives are rare.
- **TSS** (sensitivity + specificity − 1) — the SDM literature's standard;
  0 is chance, 1 is perfect, and unlike accuracy it is not fooled by imbalance.
- **Boyce index** — Spearman correlation between predicted suitability and the
  ratio of observed presences to background. **It requires no true absences**,
  which is the entire reason it belongs here: OBIS gives presences and a
  background, never a verified absence.

In [ ]:
# Rebuild the held-out block used for the final fit (same seed as notebook 03).
final_split = next(iter(splits.spatial_block_splits(
    frame.latitude, frame.longitude, n_splits=5, block_degrees=3.0,
    seed=config.RANDOM_SEED + 1)))
train_frame = frame.iloc[final_split.train]
test_frame = feature_lib.apply_thermal_niche(frame.iloc[final_split.test], niche)

predictions = {}
for name, model in models.items():
    predictions[name] = model.predict_proba(test_frame[model_columns])[:, 1]
predictions["ensemble"] = sum(
    ensemble_weights.get(name, 0.0) * scores for name, scores in predictions.items()
)
y_true = test_frame.presence.to_numpy()
print(f"scored {len(y_true)} held-out points ({y_true.sum()} presences)")

## 2. ROC and precision–recall curves

The PR curve is the more informative of the two here: with a 3:1 background
ratio, a classifier that predicts "background" everywhere already gets 75%
accuracy, and the ROC curve flatters it.

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
model_color = {name: viz.CATEGORICAL[i] for i, name in enumerate(sorted(predictions))}

for name, scores in predictions.items():
    fpr, tpr, _ = roc_curve(y_true, scores)
    axes[0].plot(fpr, tpr, color=model_color[name], label=name, linewidth=2)

    precision, recall, _ = precision_recall_curve(y_true, scores)
    axes[1].plot(recall, precision, color=model_color[name], label=name, linewidth=2)

axes[0].plot([0, 1], [0, 1], color=viz.INK_MUTED, linewidth=1)
axes[0].text(0.55, 0.48, "chance", fontsize=8, color=viz.INK_SECONDARY, rotation=33)
axes[0].legend(loc="lower right")
viz.label_axes(axes[0], title="ROC curve",
               subtitle="Optimistic under imbalance — reported for comparability with the literature.",
               xlabel="false positive rate", ylabel="true positive rate")

baseline = y_true.mean()
axes[1].axhline(baseline, color=viz.INK_MUTED, linewidth=1)
axes[1].text(0.02, baseline + 0.015, f"chance ({baseline:.2f})",
             fontsize=8, color=viz.INK_SECONDARY)
axes[1].legend(loc="upper right")
viz.label_axes(axes[1], title="Precision–recall curve",
               subtitle="The honest view: how precision decays as you demand more recall.",
               xlabel="recall", ylabel="precision")
plt.tight_layout()
plt.show()

In [ ]:
# Operating points: what would an advisory actually look like at each recall?
rows = []
for target_recall in (0.60, 0.70, 0.80, 0.90):
    threshold = metrics.threshold_for_recall(y_true, predictions["ensemble"], target_recall)
    precision, recall = metrics.precision_recall_at_threshold(
        y_true, predictions["ensemble"], threshold)
    rows.append({
        "target recall": target_recall,
        "threshold": round(threshold, 3),
        "precision": round(precision, 3),
        "achieved recall": round(recall, 3),
        "false-alarm rate": round(metrics.false_alarm_rate(
            y_true, predictions["ensemble"], threshold), 3),
    })
pd.DataFrame(rows)

This table is what a fisheries advisory service would actually negotiate over:
how many good fishing zones are you willing to miss, in exchange for how many
wasted trips? The model does not answer that — it just quantifies the trade.

## 3. The Boyce index

Worth explaining because it is unfamiliar outside ecology, and because it is the
most appropriate metric for this data.

Bin the predicted suitability range into moving windows. In each window compute
**P/E** — the ratio of the fraction of *presences* falling there to the fraction
of *background* falling there. A well-calibrated habitat model puts
monotonically more presences than background into successively higher
suitability bins, so P/E should rise with suitability. The index is the Spearman
correlation between the two.

In [ ]:
scores = predictions["ensemble"]
presence_scores = scores[y_true == 1]
background_scores = scores[y_true == 0]

low, high = float(scores.min()), float(scores.max())
width = (high - low) * 0.1
starts = np.linspace(low, high - width, 100)

midpoints, ratios = [], []
for start in starts:
    stop = start + width
    p = np.mean((presence_scores >= start) & (presence_scores <= stop))
    e = np.mean((background_scores >= start) & (background_scores <= stop))
    if e > 0:
        midpoints.append(start + width / 2)
        ratios.append(p / e)

boyce = metrics.boyce_index(presence_scores, background_scores)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(midpoints, ratios, color=viz.PRESENCE, linewidth=2.5)
ax.axhline(1.0, color=viz.INK_MUTED, linewidth=1)
ax.text(max(midpoints), 1.0, " P/E = 1 (no better than background)",
        va="center", ha="right", fontsize=8, color=viz.INK_SECONDARY)
viz.label_axes(ax, title=f"Boyce curve — index {boyce:.3f}",
               subtitle="Rising left-to-right means higher predicted suitability really does hold proportionally more presences.",
               xlabel="predicted suitability", ylabel="presence-to-background ratio (P/E)")
plt.tight_layout()
plt.show()

print(f"Boyce index: {boyce:.3f}   (1.0 = perfectly monotonic, 0 = no better than background)")

## 4. SHAP — what drives the predictions

SHAP attributes each prediction to its input features. This is what feeds the
product's Explainable AI Assistant: a per-prediction reason list rather than a
bare score.

Names come back in post-preprocessing space (one-hot categories appear
separately) because that is the space SHAP attributes in — mapping them back to
source columns would merge signals the model treats separately.

In [ ]:
importances = pd.read_csv(config.REPORTS_DIR / "fish_habitat_shap.csv")
top = importances.head(18).iloc[::-1]

fig, ax = plt.subplots(figsize=(9, 6.5))
# Single series → one colour. Emphasis marks the bathymetric group specifically,
# because that grouping is the story (see the note below).
BATHY = ("depth", "distance_to_coast", "seafloor_slope", "log_depth")
colors = [viz.ACCENT if any(b in f for b in BATHY) else viz.PRESENCE
          for f in top.feature]
ax.barh(range(len(top)), top.mean_abs_shap, color=colors, height=0.75)
ax.set_yticks(range(len(top)))
ax.set_yticklabels([f.replace("numeric__", "").replace("categorical__", "")
                    for f in top.feature], fontsize=8)
ax.grid(axis="x"); ax.grid(axis="y", visible=False)
viz.label_axes(ax, title="SHAP feature importance (LightGBM)",
               subtitle="Orange = bathymetric/coastal. They dominate — see the caveat below.",
               xlabel="mean |SHAP value|")
plt.tight_layout()
plt.show()

bathy_share = importances[importances.feature.str.contains("|".join(BATHY))].mean_abs_shap.sum()
print(f"bathymetric/coastal share of total |SHAP|: {bathy_share / importances.mean_abs_shap.sum():.1%}")

> ### An honest caveat about the top features
>
> Depth and distance-to-coast dominate. Some of that is real ecology — pelagic
> tunas and coastal sardines genuinely occupy different depth habitats.
>
> But some of it is **residual sampling geometry**. Target-group background
> sampling corrects *horizontal* bias well: background points come from the same
> places the presences do. It does not correct *depth* bias, because the
> background pool skews shallower and more coastal than the pelagic tunas that
> make up most of the presences.
>
> So part of what the model calls "deep water is good tuna habitat" may be
> "tuna records come from deep-water fisheries, and mackerel records come from
> inshore surveys". Constraining background draws to match the presence depth
> distribution is the natural next step.
>
> This is exactly the sort of thing that a good AUC would hide, which is why it
> is stated here rather than buried.

## 5. Response curves

SHAP says *which* features matter. Response curves say *in which direction* —
and that is what can be checked against known ecology.

In [ ]:
model = models["lightgbm"]
train_fold = feature_lib.apply_thermal_niche(train_frame, niche)

RESPONSE_VARS = [v for v in
                 ["depth", "distance_to_coast", "thetao", "chl", "sst_gradient", "mlotst"]
                 if v in model_columns]

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
baseline_row = train_fold[model_columns].median(numeric_only=True)

for ax, var in zip(axes.ravel(), RESPONSE_VARS):
    values = train_fold[var].dropna()
    grid = np.linspace(values.quantile(0.02), values.quantile(0.98), 60)

    # Partial dependence by hand: hold everything else at a typical value and
    # sweep one variable, so the curve isolates that variable's effect.
    probe = train_fold[model_columns].sample(
        min(400, len(train_fold)), random_state=config.RANDOM_SEED).copy()
    curve = []
    for value in grid:
        probe[var] = value
        curve.append(model.predict_proba(probe)[:, 1].mean())

    ax.plot(grid, curve, color=viz.PRESENCE, linewidth=2.5)
    # Rug of the observed distribution: shows where the curve is supported.
    ax.plot(values.sample(min(300, len(values)), random_state=0),
            np.full(min(300, len(values)), min(curve)), "|",
            color=viz.INK_MUTED, alpha=0.25, markersize=6)
    ax.set_title(var, loc="left", fontsize=10, color=viz.INK)
    ax.set_ylabel("mean predicted suitability", fontsize=8)
    if var in ("depth", "chl"):
        ax.set_xscale("log")

fig.suptitle("Response curves (partial dependence)", x=0.06, ha="left",
             fontsize=13, fontweight="semibold", color=viz.INK)
fig.text(0.06, 0.945,
         "Ticks along the bottom show where data actually exists — curve segments beyond them are extrapolation.",
         fontsize=9, color=viz.INK_SECONDARY)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

The rug ticks matter: where the data thins out, the curve is the model
extrapolating a fitted shape rather than describing observations. A dramatic
rise at the far end of a range with no ticks under it is an artefact.

## 6. MESS — where is the model extrapolating?

Multivariate Environmental Similarity Surface (Elith et al. 2010). For each
point, how similar its environment is to the training envelope.
**Negative values mean extrapolation** — at least one variable falls outside
anything the model saw, so its prediction there extends a fitted curve into
unobserved conditions.

This matters directly for climate-driven range shift: a model trained on
2000–2013 conditions, asked about a marine heatwave, is extrapolating, and the
product should say so rather than returning a confident number.

In [ ]:
numeric_columns = [c for c in model_columns
                   if pd.api.types.is_numeric_dtype(train_fold[c])]
similarity = metrics.mess(train_fold[numeric_columns], test_frame[numeric_columns])

fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))

axes[0].hist(similarity.dropna(), bins=50, color=viz.PRESENCE, edgecolor="none")
axes[0].axvline(0, color=viz.CATEGORICAL[7], linewidth=2)
axes[0].text(0, axes[0].get_ylim()[1] * 0.9, " extrapolating →",
             fontsize=8, color=viz.CATEGORICAL[7])
viz.label_axes(axes[0], title="MESS distribution on the held-out block",
               subtitle=f"{(similarity < 0).mean():.1%} of held-out points fall outside the training envelope.",
               xlabel="MESS (%)", ylabel="points")

sc = axes[1].scatter(test_frame.longitude, test_frame.latitude,
                     c=similarity, cmap=viz.DIVERGING,
                     vmin=-abs(similarity).max(), vmax=abs(similarity).max(),
                     s=18, edgecolor="none")
axes[1].set_aspect("equal"); axes[1].grid(False)
cb = fig.colorbar(sc, ax=axes[1], shrink=0.85, pad=0.02)
cb.set_label("MESS (%)", color=viz.INK_SECONDARY)
cb.outline.set_visible(False)
viz.label_axes(axes[1], title="Where extrapolation happens",
               subtitle="Red = novel conditions; predictions there deserve lower confidence.",
               xlabel="longitude (°E)", ylabel="latitude (°N)")
plt.tight_layout()
plt.show()

print(f"median MESS: {np.nanmedian(similarity):.1f}%")
print(f"extrapolating (MESS < 0): {(similarity < 0).mean():.1%} of held-out points")

## 7. Predicted suitability map

The product deliverable: a gridded habitat-suitability surface. We predict onto
the full common grid for a single month, holding the model fixed.

In [ ]:
physics = copernicus.fetch_physics(REGION, START, END, cadence="monthly")
bgc = copernicus.fetch_bgc(REGION, START, END, cadence="monthly")
bathymetry = gebco.fetch_bathymetry(REGION)

TARGET_MONTH = "2010-01"
TARGET_SPECIES_KEY = "yellowfin_tuna"

grid_frame = fusion.build_gridded_frame(
    physics.sel(time=TARGET_MONTH), bgc.sel(time=TARGET_MONTH),
    bathymetry, region=REGION, resolution=config.GRID_RESOLUTION,
)
print(f"prediction grid: {len(grid_frame)} ocean cells for {TARGET_MONTH}")

In [ ]:
from marine_ml.features import temporal

# Assemble the same feature set the model was trained on. Anything the gridded
# path cannot supply (the point-sampled prey lags) is filled with the training
# median, and the map is labelled accordingly — a missing feature silently
# defaulting to zero would distort the surface.
prediction_frame = grid_frame.copy()
prediction_frame["species_key"] = TARGET_SPECIES_KEY
prediction_frame["log_depth"] = np.log1p(prediction_frame["depth"].clip(lower=0))
prediction_frame["depth_bucket"] = pd.cut(
    prediction_frame["depth"], bins=[-np.inf, 50, 200, 1000, 3000, np.inf],
    labels=["nearshore", "shelf", "slope", "deep", "abyssal"]).astype("category")

cyclical = temporal.cyclical_time_features(prediction_frame["date"])
prediction_frame = pd.concat(
    [prediction_frame.reset_index(drop=True), cyclical.reset_index(drop=True)], axis=1)
prediction_frame["monsoon_phase"] = temporal.monsoon_phase(prediction_frame["date"])
prediction_frame = feature_lib.apply_thermal_niche(prediction_frame, niche)

medians = train_fold[model_columns].median(numeric_only=True)
missing_features = []
for column in model_columns:
    if column not in prediction_frame.columns:
        missing_features.append(column)
        prediction_frame[column] = medians.get(column, 0.0)

print(f"{len(missing_features)} features filled with training medians:")
print("  " + ", ".join(missing_features) if missing_features else "  (none)")

suitability = models["lightgbm"].predict_proba(prediction_frame[model_columns])[:, 1]
prediction_frame["suitability"] = suitability
print(f"\nsuitability range: {suitability.min():.3f} – {suitability.max():.3f}")

In [ ]:
pivot = prediction_frame.pivot_table(index="latitude", columns="longitude",
                                     values="suitability")

fig, ax = plt.subplots(figsize=(11, 7))
mesh = ax.pcolormesh(pivot.columns, pivot.index, pivot.values,
                     cmap=viz.SEQUENTIAL, vmin=0, vmax=1, shading="auto")
viz.annotate_land(ax, bathymetry, REGION)

# Overlay the actual observations for this species as a reality check.
observed = frame[(frame.species_key == TARGET_SPECIES_KEY) & (frame.presence == 1)]
ax.scatter(observed.longitude, observed.latitude, s=10,
           facecolor="none", edgecolor=viz.ACCENT, linewidth=0.8,
           label=f"observed presences (all months)")
ax.legend(loc="lower left", markerscale=1.5)

cb = fig.colorbar(mesh, ax=ax, shrink=0.85, pad=0.02)
cb.set_label("predicted habitat suitability", color=viz.INK_SECONDARY)
cb.outline.set_visible(False)

viz.label_axes(
    ax,
    title=f"Predicted habitat suitability — {TARGET_SPECIES_KEY.replace('_', ' ')}, {TARGET_MONTH}",
    subtitle="Sequential ramp: light = low suitability. Orange rings are real observations, for comparison.",
    xlabel="longitude (°E)", ylabel="latitude (°N)",
)
plt.tight_layout()
plt.show()

Read this map with the caveat from §4 in mind: the strong depth dependence means
the surface partly traces bathymetry. The observed presences (orange rings) span
all months, while the surface is one month, so they will not align perfectly —
they are a sanity check on the broad pattern, not a per-cell validation.

## 8. What this model can and cannot be trusted to do

In [ ]:
insights = pd.DataFrame([
    {"finding": "Ranks habitat better than chance on unseen water",
     "evidence": f"block-CV ROC-AUC {fold_scores.groupby('model').roc_auc.mean().max():.2f}, "
                 f"TSS {fold_scores.groupby('model').tss.mean().max():.2f}",
     "confidence": "high"},
    {"finding": "Suitability ranking is monotonic (Boyce)",
     "evidence": f"Boyce {boyce:.2f} on the held-out block",
     "confidence": "high"},
    {"finding": "Depth and distance-to-coast dominate",
     "evidence": f"{bathy_share / importances.mean_abs_shap.sum():.0%} of total |SHAP|",
     "confidence": "high (but see caveat)"},
    {"finding": "Bathymetric dominance is partly sampling geometry",
     "evidence": "background pool skews shallower than pelagic presences",
     "confidence": "suspected, not quantified"},
    {"finding": "Frontal/eddy features contribute modestly",
     "evidence": "EDA showed only a small gradient shift at presences",
     "confidence": "moderate"},
    {"finding": "No single model tier dominates",
     "evidence": "near-equal ensemble weights",
     "confidence": "high"},
])
insights

### Limitations — read before using any of this operationally

1. **Presence-only labels.** There are no verified absences anywhere in this
   pipeline. Every "absence" is a constructed background point, and every metric
   is conditional on that construction being reasonable.

2. **Residual depth bias in the background.** Target-group sampling fixes
   horizontal bias, not depth bias. Part of the model's strongest signal is
   probably an artefact of which fisheries generate which records. Fixing this —
   stratifying background draws by depth — is the highest-value next step.

3. **~1,100 presences across five species.** Small, and unevenly split. The
   coastal species have well under 200 records each; their species-specific
   behaviour is weakly determined.

4. **2000–2013 only.** The model has never seen recent conditions. Given
   climate-driven range shift, extrapolating it to the present is exactly the
   situation MESS exists to flag.

5. **Monthly resolution.** Fish move on shorter timescales than a month. This
   supports strategic questions ("where is habitat generally good in January?"),
   not tactical ones ("where should a boat go tomorrow?").

6. **Correlation, not mechanism.** The model finds environmental associations.
   It does not know about fishing pressure, prey species composition, or
   behaviour, so it cannot explain *why* a zone is good.

### Recommended next steps, in order of expected value

1. **Depth-stratified background sampling** — directly attacks limitation 2,
   which is the largest known weakness.
2. **Add CMFRI landings data** as an independent label source, and validate
   against INCOIS's operational PFZ advisories — external validation, never
   training.
3. **Global Fishing Watch effort as a covariate and out-of-sample check only.**
   Never as a label: effort follows fish, so training on it would just reproduce
   existing fishing patterns.
4. **Tier 3 spatio-temporal models** (CNN/ViT over stacked rasters) once the
   label quality above is addressed. Model capacity is not currently the
   binding constraint — label quality is.